# E004 training on Kaggle -- fold 1

CODING_SPEC_03_FP_AND_FOLDS.md: same model as E002 (resnet34/imagenet U-Net)
with EMA of weights (decay 0.999) and checkpoint selection by instance PQ
(every 1000 iterations, on a fixed 40-image subset of fold 1's val
split) instead of pixel Dice, 4000 iterations. One of five near-identical
notebooks (fold 0-4), each pushed and run as its own Kaggle kernel so the
five folds can train in parallel across separate Kaggle sessions.

Designed to run unattended top to bottom (**Save & Run All**), with Internet
**on** (for `pip install` and the ImageNet encoder weights).

Setup (manual, one time, same package dataset as E002 -- rebuild/push it
first with `scripts/package_kaggle.py` if `configs/E004-f1.yaml` or any
`src/`/`scripts/` change isn't in it yet):
1. Attach the `potcharapanchumrak/solar-k-package` dataset and the `filament-segmentation-2026` competition.
2. Turn on a GPU accelerator and Internet access.
3. If resuming from a previous notebook version's output, attach that
   version's output as a dataset too and set `RESUME_FROM` below to its
   `last.pt` path; otherwise leave it `None`.

Checkpoints, logs, OOF/test probability maps all write directly to
`/kaggle/working/outputs` (the persisted working directory) -- no
copy-at-the-end step needed. Never put credentials in this notebook or in
the uploaded package.

In [ ]:
import os
import time

# On this Kaggle layout, attached sources are nested under /kaggle/input/datasets/
# and /kaggle/input/competitions/ (not flat /kaggle/input/<slug>/).
PKG_DIR = '/kaggle/input/datasets/potcharapanchumrak/solar-k-package'

# Kaggle has shown intermittent FileNotFoundError here even though the
# dataset_sources attachment is correctly registered (confirmed via
# `kaggle kernels pull -m`) -- a short wait costs little and occasionally
# helps, but a 120s wait was tried and the mount still never appeared, so
# this doesn't retry long: fail fast and let the external push-retry loop
# (scripts/gen_e004_kaggle_artifacts.py's caller) try a fresh session
# instead of burning GPU minutes on a wait that isn't fixing it.
for attempt in range(10):
    if os.path.isdir(PKG_DIR):
        break
    print(f'{PKG_DIR} not mounted yet, waiting (attempt {attempt + 1}/10)...')
    time.sleep(3)
else:
    raise FileNotFoundError(f'{PKG_DIR} never appeared after 30s -- check the dataset is attached')

os.chdir(PKG_DIR)
print(os.listdir('.'))

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os

# The competition mount is .../MAGFiLO_1.0_Kaggle_2026/{train,test}/..., but
# solar_k.io expects <SOLAR_K_DATA>/raw/{train,test}/... (matching the local
# data/raw/ layout) -- bridge the one missing "raw" level with a symlink
# rather than changing io.py's path assumptions (which also serve local runs).
# splits/folds_v1.csv similarly comes from the uploaded package, not the
# competition mount, so it needs its own symlink alongside raw/.
COMP_ROOT = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
PKG_SPLITS = os.path.join(PKG_DIR, 'data', 'splits')
DATA_LINK_ROOT = '/kaggle/working/data_root'
os.makedirs(DATA_LINK_ROOT, exist_ok=True)
for name, target in [('raw', COMP_ROOT), ('splits', PKG_SPLITS)]:
    link = os.path.join(DATA_LINK_ROOT, name)
    if not os.path.islink(link):
        os.symlink(target, link)

os.environ['SOLAR_K_DATA'] = DATA_LINK_ROOT

# Checkpoints/logs/probs go straight into the persisted working directory.
os.environ['SOLAR_K_OUTPUT'] = '/kaggle/working/outputs'

# Preprocessing cache on fast, writable local disk (the competition mount is read-only).
os.environ['SOLAR_K_CACHE'] = '/kaggle/temp/solar_k_cache'

# Set to a last.pt path (e.g. under /kaggle/input/<prior-version-dataset>/outputs/E004-f{fold}/checkpoints/last.pt)
# to continue a previous notebook version's run instead of starting fresh.
RESUME_FROM = None

# E004's 4000 iterations finish well under this; kept as a safety cap in
# case a Kaggle session is slower than expected.
MAX_HOURS = 8.5

In [ ]:
import subprocess

cmd = ['python', 'scripts/train.py', '--config', 'configs/E004-f1.yaml', '--max_hours', str(MAX_HOURS)]
if RESUME_FROM:
    cmd += ['--resume', RESUME_FROM]

# One invocation covers the whole run: training (EMA weights, checkpoint
# selection by instance PQ every 1000 iters on a 40-image subset, see
# configs/E004-f1.yaml), then -- in the same process -- flip-TTA OOF
# (fold 1 val) and test-set probability maps saved as compressed uint8
# PNGs. No postprocessing/threshold tuning happens here; that runs locally
# against these saved probs (scripts/run_e005_eval.py, once all 5 folds are in).
subprocess.run(cmd, check=True)